# Event Graph Little's Law Verification

This notebook demonstrates:
1. Defining an **Event Graph** model using JSON specification
2. Converting JSON to SimASM using the **converter**
3. Running a **Little's Law** verification experiment

## Little's Law

For an M/M/5 queue with:
- λ = 0.8 (arrival rate = 1/1.25)
- μ = 1.0 (service rate = 1/1.0)
- ρ = λ/(5μ) = 0.16 (per-server utilization)

Little's Law states: **L = λW**

Where:
- L = average number in system
- W = average time in system
- λ = arrival rate

In [1]:
# Install simasm (uncomment in Colab)
# !pip install simasm

import simasm
print(f"SimASM version: {simasm.__version__}")

SimASM version: 0.2.6


## 1. Define Event Graph JSON Specification

The Event Graph formalism (Schruben 1983) uses:
- **Vertices**: State-changing events
- **Scheduling edges**: Conditional event scheduling with delays
- **State variables**: System state (queue length, server availability)

Algebraic specification: **S = (F, C, T, Γ, G)**
- F: State transition functions
- C: Edge conditions
- T: Edge delays
- Γ: Priorities
- G: Graph structure

In [3]:
# Define the M/M/5 Event Graph as JSON
mm5_eg_json = {
  "model_name": "mm5_eg",
  "description": "M/M/5 Queue using Event Graph formalism (Schruben 1983)",

  "state_variables": {
    "q": {"type": "Nat", "initial": 0, "description": "Queue length (customers waiting)"},
    "p": {"type": "Nat", "initial": 5, "description": "Available servers (idle)"}
  },

  "parameters": {
    "k": {"type": "Nat", "value": 5, "description": "Number of servers (capacity)"},
    "iat_mean": {"type": "Real", "value": 1.25, "description": "Inter-arrival time mean (1/λ)"},
    "ist_mean": {"type": "Real", "value": 1.0, "description": "Service time mean (1/μ)"},
    "sim_end_time": {"type": "Real", "value": 1000.0, "description": "Simulation end time"}
  },

  "random_streams": {
    "T_a": {
      "distribution": "exponential",
      "params": {"mean": "iat_mean"},
      "stream_name": "arrivals"
    },
    "T_s": {
      "distribution": "exponential",
      "params": {"mean": "ist_mean"},
      "stream_name": "service"
    }
  },

  "vertices": [
    {
      "name": "Arrive",
      "state_change": "q := q + 1",
      "description": "Customer arrival - increment queue"
    },
    {
      "name": "AttemptToStart",
      "state_change": "",
      "description": "Decision node - attempt to start service (no state change)"
    },
    {
      "name": "Start",
      "state_change": "q := q - 1; p := p - 1",
      "description": "Service start - decrement queue and available servers"
    },
    {
      "name": "Finish",
      "state_change": "p := p + 1",
      "description": "Service completion - release server"
    }
  ],

  "scheduling_edges": [
    {
      "from": "Arrive",
      "to": "Arrive",
      "delay": "T_a",
      "condition": "true",
      "priority": 0,
      "description": "Self-loop: schedule next arrival"
    },
    {
      "from": "Arrive",
      "to": "AttemptToStart",
      "delay": 0,
      "condition": "true",
      "priority": 1,
      "description": "Always attempt to start after arrival"
    },
    {
      "from": "AttemptToStart",
      "to": "Start",
      "delay": 0,
      "condition": "p > 0 and q > 0",
      "priority": 0,
      "description": "Start only if server available AND customer waiting"
    },
    {
      "from": "Start",
      "to": "Finish",
      "delay": "T_s",
      "condition": "true",
      "priority": 0,
      "description": "Schedule service completion"
    },
    {
      "from": "Finish",
      "to": "AttemptToStart",
      "delay": 0,
      "condition": "q > 0",
      "priority": 0,
      "description": "If queue not empty, attempt to start next customer"
    }
  ],

  "cancelling_edges": [],

  "initial_events": [
    {
      "event": "Arrive",
      "time": "T_a"
    }
  ],

  "stopping_condition": "sim_clocktime >= sim_end_time",

  "observables": {
    "queue_length": {
      "expression": "q",
      "return_type": "Nat",
      "description": "Number in queue"
    },
    "servers_busy": {
      "expression": "k - p",
      "return_type": "Nat",
      "description": "Number of busy servers"
    },
    "in_system": {
      "expression": "q + (k - p)",
      "return_type": "Nat",
      "description": "Total customers in system"
    }
  }
}

# Save JSON to file for the converter
import json
with open("mm5_eg.json", "w") as f:
    json.dump(mm5_eg_json, f, indent=2)
print("Saved mm5_eg.json")

Saved mm5_eg.json


## 2. Convert JSON to SimASM Model

Use the **converter** to generate SimASM code from the JSON specification.

The converter implements the **Next-Event Time-Advance Algorithm** (Law 2013):
1. **Initialization**: Set clock=0, STATE=STATE_0, FEL=FEL_0
2. **Timing**: Pop earliest event from FEL, advance clock
3. **Event routine**: Execute f_v, process scheduling edges with c_e and t_e
4. Repeat until termination condition

In [6]:
%%simasm convert

convert mm5_eg:
    source: "mm5_eg.json"
    formalism: event_graph
    register: "mm5_eg"
    print: true
endconvert

## 3. Run Little's Law Experiment

We'll collect statistics to verify Little's Law:
- **L** = average number in system
- **L_q** = average number in queue
- **L_s** = average number in service
- **ρ** = server utilization

For M/M/5 with λ=0.8, μ=1.0:
- ρ = λ/(5μ) = 0.16 (per server)
- L_s = 5ρ = 0.8 (avg servers busy)

In [7]:
%%simasm experiment
// Little's Law Experiment for ACD M/M/5 Queue

experiment LittlesLawEG:
    model := "mm5_eg"
    
    replication:
        count: 30
        warm_up_time: 100.0
        run_length: 1000.0
        seed_strategy: "incremental"
        base_seed: 12345
    endreplication
    
    statistics:
        // L: Average number in system
        stat L_system: time_average
            expression: "q + (k - p)"
        endstat
        
        // L_q: Average number in queue
        stat L_queue: time_average
            expression: "q"
        endstat
        
        // L_s: Average number in service (busy servers)
        stat L_service: time_average
            expression: "k - p"
        endstat

        // Server utilization (fraction of capacity used)
        stat rho_utilization: time_average
            expression: "(k - p) / k"
        endstat
        
    endstatistics
endexperiment


  Replication 30/30...


Statistic,Mean,Std Dev,95% CI
L_system,0.8054,0.0389,"[0.7908, 0.8199]"
L_queue,0.0005,0.0008,"[0.0002, 0.0008]"
L_service,0.8049,0.0389,"[0.7904, 0.8194]"
rho_utilization,0.1610,0.0078,"[0.1581, 0.1639]"


ExperimentResult(config=ExperimentConfig(name='LittlesLawEG', model_path='C:\\Users\\steve\\AppData\\Local\\Temp\\simasm_59vwkg1k\\mm5_eg.simasm', model_source=None, replications=ReplicationSettings(count=30, warmup=100.0, length=1000.0, base_seed=12345), statistics=[StatisticConfig(name='L_system', type='time_average', domain=None, expr='q + (k - p)', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='L_queue', type='time_average', domain=None, expr='q', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='L_service', type='time_average', domain=None, expr='k - p', interval=None, condition=None, aggregation='average', start_expr=None, end_expr=None, entity_domain=None), StatisticConfig(name='rho_utilization', type='time_average', domain=None, expr='(k - p) / k', interval=None, condition=None, aggregation='average', start_expr=None, e

## Summary

This notebook demonstrated:

1. **JSON → SimASM Conversion**: Used the Event Graph converter to generate executable SimASM code from a JSON specification

2. **Little's Law Verification**: Confirmed that the generated model produces statistically valid results:
   - Conservation: L = L_q + L_s
   - Utilization: L_s ≈ λ/μ

3. **Event Graph Formalism**: The algebraic specification S = (F, C, T, Γ, G) maps cleanly to the JSON format and generates correct SimASM code.

### References

- Schruben, L.W. (1983). Simulation Modeling with Event Graphs. Communications of the ACM.
- Law, A.M. (2013). Simulation Modeling and Analysis (5th ed.). McGraw-Hill.
- Little, J.D.C. (1961). A Proof for the Queuing Formula: L = λW. Operations Research.